# Assessing Data Accessibility from Files Downloaded from the SEEA 2024 Global Assessment
The data accessibility assessment uses files downloaded from the SEEA 2024 Global Assessment: https://seea.un.org/content/2024-global-assessment
.

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import tempfile
import unicodedata
import zipfile
from pathlib import Path

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "bs4": "beautifulsoup4",
    "pypdf": "pypdf",
    "docx": "python-docx",
    "xlrd": "xlrd",
}

for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import pandas as pd
from bs4 import BeautifulSoup
from docx import Document
from pypdf import PdfReader


SOURCE_XLSX = "global_assessment_for_website_2024_final.xlsx"
SHEET_NAME = "2024 Global Assessment results"
SEEA_FOLDER = "seea_file"
CONFIG_JSON = "seea_keyword_config.json"

OUTPUT_XLSX = "seea_account_availability_assessment.xlsx"
OUTPUT_WIDE_CSV = "seea_account_availability_assessment.csv"
OUTPUT_LONG_CSV = "seea_account_availability_assessment_long.csv"

MAX_TEXT_CHARS = 2_000_000
MAX_PDF_PAGES = 40
MAX_ZIP_FILES = 300

ACCOUNT_TYPES = [
    "Material flow accounts",
    "Economy wide material flow accounts",
    "Physical supply and use tables for water",
    "Monetary supply and use tables for water",
    "Physical asset accounts for water",
    "Water emission accounts",
    "Physical supply and use tables for energy",
    "Monetary supply and use tables for energy",
    "Physical asset accounts for mineral and energy resources",
    "Monetary asset accounts for mineral and energy resources",
    "Air emission accounts",
    "Physical asset accounts for land",
    "Monetary asset accounts for land",
    "Physical asset accounts for timber resources",
    "Monetary asset accounts for timber resources",
    "Physical asset accounts for aquatic resources",
    "Monetary asset accounts for aquatic resources",
    "Asset accounts for other biological resources",
    "Waste accounts",
    "Environmental protection expenditure accounts",
    "Resource management expenditure accounts",
    "Environmental goods and services accounts",
    "Taxes accounts",
    "Environmental subsidies accounts",
    "Agriculture, Forestry and Fisheries accounts",
    "Ecosystem extent accounts",
    "Ecosystem condition accounts",
    "Physical ecosystem services accounts",
    "Monetary ecosystem services accounts",
    "Ecosystem asset accounts",
    "Ocean accounts",
    "Species accounts",
    "Carbon accounts",
    "Urban accounts",
    "Protected areas accounts",
    "Forest accounts",
]

FALSE_POSITIVE_PATTERNS = [
    r"\buser account\b",
    r"\blogin account\b",
    r"\bsign in account\b",
    r"\bpersonal account\b",
    r"\bbank account\b",
    r"\bpayment account\b",
    r"\baccount settings\b",
    r"\bmy account\b",
    r"\baccounting firm\b",
    r"\bfinancial accounting\b",
]


def normalize_text(value):
    value = str(value or "")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[_/\\|:;,\.\-\(\)\[\]\{\}]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def clean_country_name(value):
    value = str(value or "").strip()
    value = re.sub(r"[^A-Za-z0-9._ -]+", "_", value)
    value = re.sub(r"\s+", "_", value).strip(" ._")
    return value or "unknown_country"


def make_snippet(text, keyword, width=260):
    normalized = normalize_text(text)
    needle = normalize_text(keyword)
    index = normalized.find(needle)
    if index < 0:
        return normalized[:width]
    start = max(0, index - width // 2)
    end = min(len(normalized), index + len(needle) + width // 2)
    return normalized[start:end]


def has_false_positive_context(snippet):
    normalized = normalize_text(snippet)
    return any(re.search(pattern, normalized) for pattern in FALSE_POSITIVE_PATTERNS)


def sniff_extension(path):
    suffix = path.suffix.lower()
    try:
        data = path.read_bytes()[:4096]
    except Exception:
        return suffix
    if data.startswith(b"%PDF"):
        return ".pdf"
    if data.startswith(b"PK\x03\x04"):
        return ".zip"
    if data.startswith(b"\xD0\xCF\x11\xE0"):
        return ".xls"
    sample = data.lower()
    if b"<html" in sample or b"<!doctype html" in sample:
        return ".html"
    return suffix


def read_text_file(path):
    for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return path.read_text(encoding=encoding, errors="replace")[:MAX_TEXT_CHARS]
        except Exception:
            continue
    return path.read_bytes().decode("utf-8", errors="replace")[:MAX_TEXT_CHARS]


def extract_html_text(path):
    raw = read_text_file(path)
    soup = BeautifulSoup(raw, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    headings = " ".join(h.get_text(" ", strip=True) for h in soup.find_all(re.compile("^h[1-3]$")))
    body = soup.get_text(" ", strip=True)
    return f"{title}\n{headings}\n{body}"[:MAX_TEXT_CHARS], "html"


def extract_pdf_text(path):
    reader = PdfReader(str(path))
    parts = []
    for page in reader.pages[:MAX_PDF_PAGES]:
        try:
            parts.append(page.extract_text() or "")
        except Exception as exc:
            parts.append(f" [page extraction error: {exc}] ")
    return "\n".join(parts)[:MAX_TEXT_CHARS], "pdf text"


def extract_docx_text(path):
    doc = Document(str(path))
    parts = [paragraph.text for paragraph in doc.paragraphs]
    for table in doc.tables:
        for row in table.rows:
            parts.append(" ".join(cell.text for cell in row.cells))
    return "\n".join(parts)[:MAX_TEXT_CHARS], "Word document"


def extract_csv_text(path):
    try:
        df = pd.read_csv(path, dtype=str, encoding="utf-8", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, dtype=str, encoding="latin-1", on_bad_lines="skip")
    return df.fillna("").astype(str).to_csv(index=False)[:MAX_TEXT_CHARS], "CSV"


def extract_excel_text(path):
    parts = []
    excel = pd.ExcelFile(path)
    for sheet in excel.sheet_names:
        try:
            df = pd.read_excel(path, sheet_name=sheet, dtype=str, header=None)
            parts.append(f"Sheet: {sheet}")
            parts.append(df.fillna("").astype(str).to_csv(index=False, header=False))
        except Exception as exc:
            parts.append(f"Sheet read error {sheet}: {exc}")
        if sum(len(part) for part in parts) > MAX_TEXT_CHARS:
            break
    return "\n".join(parts)[:MAX_TEXT_CHARS], "Excel sheet"


def extract_zip_text(path):
    parts = []
    read_count = 0
    with tempfile.TemporaryDirectory() as temp_dir:
        with zipfile.ZipFile(path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir() or read_count >= MAX_ZIP_FILES:
                    continue
                member_name = member.filename.replace("\\", "/")
                if member_name.startswith("/") or ".." in Path(member_name).parts:
                    continue
                try:
                    archive.extract(member, temp_dir)
                    inner_path = Path(temp_dir) / member.filename
                    if inner_path.is_file():
                        inner_text, inner_location = extract_file_text(inner_path)
                        parts.append(f"\n--- File inside ZIP: {member.filename} [{inner_location}] ---\n{inner_text}")
                        read_count += 1
                except Exception as exc:
                    parts.append(f"Unreadable file inside ZIP {member.filename}: {exc}")
                if sum(len(part) for part in parts) > MAX_TEXT_CHARS:
                    break
    if not parts:
        raise RuntimeError("zip file contained no readable supported files")
    return "\n".join(parts)[:MAX_TEXT_CHARS], "ZIP archive"


def extract_file_text(path):
    suffix = sniff_extension(path)
    if suffix in (".html", ".htm"):
        return extract_html_text(path)
    if suffix == ".txt":
        return read_text_file(path), "text"
    if suffix == ".csv":
        return extract_csv_text(path)
    if suffix in (".xlsx", ".xlsm", ".xls", ".ods"):
        return extract_excel_text(path)
    if suffix == ".docx":
        return extract_docx_text(path)
    if suffix == ".pdf":
        return extract_pdf_text(path)
    if suffix == ".zip":
        return extract_zip_text(path)
    raise RuntimeError(f"unsupported file type: {path.suffix.lower() or '[no extension]'}")


def find_country_folder(country):
    base = Path(SEEA_FOLDER)
    expected = base / clean_country_name(country)
    if expected.exists():
        return expected
    normalized_country = normalize_text(country)
    if base.exists():
        for folder in base.iterdir():
            if folder.is_dir() and normalize_text(folder.name) == normalized_country:
                return folder
    return expected


def account_type_base_variants(account_type):
    variants = {
        account_type,
        account_type.lower(),
        account_type.replace("accounts", "account"),
        account_type.replace("Accounts", "Account"),
        account_type.replace("tables", "table"),
        account_type.replace("Tables", "Table"),
    }
    return {item for item in variants if item}


def generated_keyword_phrases(country, account_type, config):
    country_languages = config["country_languages"].get(country, ["English"])
    account_words_by_language = config["account_words_by_language"]
    topic_translations = config["topic_translations"].get(account_type, {})
    keywords = set(account_type_base_variants(account_type))

    for language in country_languages:
        account_words = account_words_by_language.get(language, [])
        topics = topic_translations.get(language, [])
        for topic in topics:
            for account_word in account_words:
                keywords.add(f"{topic} {account_word}")
                keywords.add(f"{account_word} {topic}")
                keywords.add(f"{account_word} for {topic}")
                keywords.add(f"{account_word} of {topic}")
                if language in {"Spanish", "French", "Portuguese", "Italian", "Romanian"}:
                    keywords.add(f"{account_word} de {topic}")
                    keywords.add(f"{account_word} del {topic}")
                    keywords.add(f"{account_word} des {topic}")
                    keywords.add(f"{account_word} dos {topic}")
                if language in {"German", "Dutch", "Danish", "Norwegian", "Swedish"}:
                    compact_topic = topic.replace(" ", "")
                    keywords.add(f"{compact_topic}{account_word}")

    return sorted(keywords)


TOPIC_STOPWORDS = {
    "account",
    "accounts",
    "accounting",
    "physical",
    "monetary",
    "asset",
    "assets",
    "stock",
    "stocks",
    "resource",
    "resources",
    "for",
    "and",
    "or",
    "of",
    "the",
    "in",
    "to",
    "by",
    "table",
    "tables",
    "supply",
    "use",
    "using",
    "with",
    "sector",
    "terms",
    "unit",
    "units",
    "data",
    "hybrid",
}


def find_match(
    text,
    normalized_text,
    filename,
    normalized_filename,
    keyword_rows,
    account_type,
    country,
    config,
    location_type,
):
    sources = [("filename", filename, normalized_filename), ("content", text, normalized_text)]
    for location, source_text, normalized_source in sources:
        for row in keyword_rows:
            normalized_keyword = row["normalized_keyword"]
            if normalized_keyword and normalized_keyword in normalized_source:
                snippet = make_snippet(source_text, row["keyword_phrase"])
                if has_false_positive_context(snippet):
                    continue
                return {
                    "matched_keyword": row["keyword_phrase"],
                    "match_location": location,
                    "snippet": snippet,
                }
        if location == "content" and location_type == "pdf text":
            toc_match = find_pdf_toc_account_match(source_text, account_type, country, config)
            if toc_match:
                return toc_match
    return None


def find_pdf_toc_account_match(source_text, account_type, country, config):
    account_words = get_account_words_for_country(country, config)
    topic_token_sets = get_account_topic_token_sets(account_type, country, config)
    if not account_words or not topic_token_sets:
        return None

    for line in extract_pdf_toc_candidate_lines(source_text):
        normalized_line = normalize_text(line)
        if not normalized_line:
            continue
        if not has_account_word(normalized_line, account_words):
            continue
        if has_false_positive_context(line):
            continue

        line_tokens = set(normalized_line.split())
        if not any(topic_tokens_match(line_tokens, token_set) for token_set in topic_token_sets):
            continue

        return {
            "matched_keyword": f"PDF TOC/front matter: {line.strip()}",
            "match_location": "pdf table of contents/front matter",
            "snippet": line.strip(),
        }
    return None


def get_account_words_for_country(country, config):
    languages = ["English", *config["country_languages"].get(country, [])]
    words = set()
    for language in languages:
        for word in config["account_words_by_language"].get(language, []):
            normalized_word = normalize_text(word)
            if normalized_word:
                words.add(normalized_word)
    return words


def has_account_word(normalized_line, account_words):
    line_tokens = set(normalized_line.split())
    if line_tokens & account_words:
        return True

    account_phrases = {
        "account",
        "accounts",
        "accounting",
        "stock accounts",
        "flow accounts",
        "asset accounts",
        "resource accounts",
    }
    return any(phrase in normalized_line for phrase in account_phrases)


def get_account_topic_token_sets(account_type, country, config):
    languages = ["English", *config["country_languages"].get(country, [])]
    phrases = [account_type]
    topic_map = config["topic_translations"].get(account_type, {})
    for language in languages:
        phrases.extend(topic_map.get(language, []))

    official_tokens = phrase_to_topic_tokens(account_type)
    token_sets = []
    for phrase in phrases:
        phrase_tokens = phrase_to_topic_tokens(phrase)
        if not phrase_tokens:
            continue

        # Do not let a one-word English subset such as only "energy" trigger a
        # compound account type whose official name also requires "mineral".
        if (
            len(official_tokens) > 1
            and len(phrase_tokens) == 1
            and phrase_tokens.issubset(official_tokens)
        ):
            continue

        if phrase_tokens and phrase_tokens not in token_sets:
            token_sets.append(phrase_tokens)
    return token_sets


def phrase_to_topic_tokens(phrase):
    normalized_phrase = normalize_text(phrase)
    tokens = set()
    for token in normalized_phrase.split():
        if len(token) < 3:
            continue
        if token in TOPIC_STOPWORDS:
            continue
        tokens.add(token)
    return tokens


def topic_tokens_match(line_tokens, topic_tokens):
    if not topic_tokens:
        return False

    overlap = line_tokens & topic_tokens
    if len(topic_tokens) == 1:
        return len(overlap) == 1
    if len(topic_tokens) == 2:
        return len(overlap) == 2

    required = max(2, int(round(len(topic_tokens) * 0.7)))
    return len(overlap) >= required


def extract_pdf_toc_candidate_lines(source_text):
    raw_lines = [line.strip() for line in str(source_text or "").splitlines()]
    lines = [line for line in raw_lines if line]
    normalized_lines = [normalize_text(line) for line in lines]

    toc_indices = [
        index
        for index, line in enumerate(normalized_lines[:250])
        if "table of contents" in line or line == "contents" or "list of tables" in line
    ]

    selected = []
    if toc_indices:
        for index in toc_indices:
            selected.extend(lines[index : index + 140])
    else:
        selected.extend(lines[:220])

    cleaned = []
    for line in selected:
        # Remove common dot leaders and terminal page numbers while preserving section text.
        line = re.sub(r"\.{3,}\s*\d+\s*$", "", line).strip()
        line = re.sub(r"\s+\d+\s*$", "", line).strip()
        if len(line) < 6:
            continue
        cleaned.append(line)
    return cleaned


def autosize_excel(writer):
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        for col_cells in worksheet.columns:
            col_letter = col_cells[0].column_letter
            max_len = 0
            for cell in col_cells[:2000]:
                value = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(value))
            worksheet.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 60)


def main():
    with open(CONFIG_JSON, "r", encoding="utf-8") as file:
        config = json.load(file)

    source_df = pd.read_excel(SOURCE_XLSX, sheet_name=SHEET_NAME, header=1)
    source_df["Country or area"] = source_df["Country or area"].astype(str).str.strip()

    keyword_cache = {}
    keyword_dictionary_rows = []
    for country in source_df["Country or area"]:
        for account_type in ACCOUNT_TYPES:
            phrases = generated_keyword_phrases(country, account_type, config)
            rows = [
                {
                    "country": country,
                    "account_type": account_type,
                    "language": "|".join(config["country_languages"].get(country, ["English"])),
                    "keyword_phrase": phrase,
                    "normalized_keyword": normalize_text(phrase),
                }
                for phrase in phrases
            ]
            keyword_cache[(country, account_type)] = rows
            keyword_dictionary_rows.extend(rows)

    matrix_rows = []
    evidence_rows = []
    error_rows = []
    long_rows = []
    files_scanned = 0

    for _, excel_row in source_df.iterrows():
        country = excel_row["Country or area"]
        country_folder = find_country_folder(country)
        files = [path for path in country_folder.rglob("*") if path.is_file()] if country_folder.exists() else []
        available = {account_type: 0 for account_type in ACCOUNT_TYPES}
        matrix_row = {"Country or area": country}

        for account_type in ACCOUNT_TYPES:
            reported = 1 if str(excel_row.get(account_type, "")).strip().lower() == "x" else 0
            matrix_row[f"{account_type}_reported"] = reported
            matrix_row[f"{account_type}_available"] = 0
            matrix_row[f"{account_type}_nonreport_avail"] = 0

        for file_path in files:
            files_scanned += 1
            try:
                text, location_type = extract_file_text(file_path)
            except Exception as exc:
                error_rows.append({"country": country, "file_path": str(file_path), "error_message": str(exc)})
                continue

            filename = file_path.name
            normalized_filename = normalize_text(filename)
            normalized_text = normalize_text(text)
            for account_type in ACCOUNT_TYPES:
                if available[account_type] == 1:
                    continue
                match = find_match(
                    text,
                    normalized_text,
                    filename,
                    normalized_filename,
                    keyword_cache[(country, account_type)],
                    account_type,
                    country,
                    config,
                    location_type,
                )
                if match:
                    available[account_type] = 1
                    evidence_rows.append({
                        "country": country,
                        "account_type": account_type,
                        "matched_keyword": match["matched_keyword"],
                        "matched_file": filename,
                        "file_path": str(file_path),
                        "match_location": (
                            "filename"
                            if match["match_location"] == "filename"
                            else match["match_location"]
                            if match["match_location"] != "content"
                            else location_type
                        ),
                        "snippet": match["snippet"],
                    })

        for account_type in ACCOUNT_TYPES:
            reported = matrix_row[f"{account_type}_reported"]
            avail = available[account_type]
            nonreport = 1 if avail == 1 and reported == 0 else 0
            matrix_row[f"{account_type}_available"] = avail
            matrix_row[f"{account_type}_nonreport_avail"] = nonreport
            long_rows.append({
                "country": country,
                "account_type": account_type,
                "reported": reported,
                "available": avail,
                "nonreport_avail": nonreport,
            })

        matrix_rows.append(matrix_row)
        print(f"{country}: files={len(files)} available_accounts={sum(available.values())}")

    matrix_df = pd.DataFrame(matrix_rows)
    evidence_df = pd.DataFrame(evidence_rows)
    errors_df = pd.DataFrame(error_rows)
    keyword_df = pd.DataFrame(keyword_dictionary_rows).drop(columns=["normalized_keyword"], errors="ignore")
    long_df = pd.DataFrame(long_rows)

    matrix_df.to_csv(OUTPUT_WIDE_CSV, index=False, encoding="utf-8-sig")
    long_df.to_csv(OUTPUT_LONG_CSV, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        matrix_df.to_excel(writer, sheet_name="availability_matrix", index=False)
        evidence_df.to_excel(writer, sheet_name="availability_evidence", index=False)
        errors_df.to_excel(writer, sheet_name="file_read_errors", index=False)
        keyword_df.to_excel(writer, sheet_name="keyword_dictionary", index=False)
        autosize_excel(writer)

    print("\nDone.")
    print(f"Countries processed: {len(matrix_rows)}")
    print(f"Files scanned: {files_scanned}")
    print(f"Evidence rows: {len(evidence_rows)}")
    print(f"File read errors: {len(error_rows)}")
    print(f"Excel output: {os.path.abspath(OUTPUT_XLSX)}")
    print(f"Wide CSV output: {os.path.abspath(OUTPUT_WIDE_CSV)}")
    print(f"Long CSV output: {os.path.abspath(OUTPUT_LONG_CSV)}")


if __name__ == "__main__":
    main()
